In [1]:
import numpy as np
import xarray as xr

import etunc.config as cfg
import etunc.temporal as temporal
import etunc.load.obs as lo
import etunc.load.ilamb as il

In [2]:
FAMILY = "obs"

TIME_SLICE = cfg.TIME_SLICE
TARGET_RES = cfg.TARGET_RES
MIN_YEARS = cfg.MIN_YEARS  # minimum complete years shared by the four products of a combination

FACTORS = ("et", "lai", "pr", "rn")  # order of the products in a combination

# Products of each variable: prep.obs of config/pipeline.yml, {label: {dataset, version, variable, freq, source}}.
# source "ilamb": il.load_native with the label; null: lo.load_obs with dataset, variable, version and freq
PRODUCTS = {
    "et": cfg.OBS_ET_PRODUCTS,
    "lai": cfg.OBS_LAI_PRODUCTS,
    "pr": cfg.OBS_PR_PRODUCTS,
    "rn": cfg.OBS_RN_PRODUCTS,
}
# Frequencies of native files that steps 1-4 handle: half-month (GIMMS LAI4g) is averaged to monthly first
NATIVE_FREQS = ("monthly", "yearly", "half-month")


products = [(v, p, spec) for v in FACTORS for p, spec in PRODUCTS[v].items()]
for v, p, spec in products:
    if spec["source"] == "ilamb" and p not in il.PRODUCTS[v]:
        raise ValueError(f"prep.obs.{v}_products: {p!r} has source ilamb but is not an ILAMB {v} product "
                            f"({list(il.PRODUCTS[v])})")
    if spec["source"] is None and spec["freq"] not in NATIVE_FREQS:
        raise ValueError(f"prep.obs.{v}_products: {p!r} has freq {spec['freq']!r}; native files must be "
                            f"one of {NATIVE_FREQS}")

In [3]:
# ------------------------------------------------------------------
# Steps 1-8, once per product: annual means on the target grid
# ------------------------------------------------------------------
ann = {v: {} for v in FACTORS}  # {variable: {product: (year, lat, lon)}}
native_res = {}  # {(variable, product): native resolution}
for v, p, spec in products:
    label = f"{v}/{p}"
    print(f"\n=== {label} ===")
    from_ilamb = spec["source"] == "ilamb"

    # 1-2. Monthly (or yearly) native field over TIME_SLICE
    try:
        if from_ilamb:
            da = il.load_native(v, p, TIME_SLICE)
        else:
            da = lo.load_obs(spec["dataset"], spec["variable"], TIME_SLICE, version=spec["version"],
                                freq=spec["freq"])
    except FileNotFoundError as err:
        print(f"{err}; skipping {label}")
        continue
    # Half-monthly composites (GIMMS LAI4g): monthly means, the equal-weight mean of each month's valid halves
    if not from_ilamb and spec["freq"] == "half-month":
        da = temporal.half_monthly_to_monthly(da)

    ann[v][p] = da


=== et/PMLv2.2a_MODIS ===

=== et/PMLv2.2b ===

=== et/PMLv2.2c ===

=== et/GLEAMv4.3a ===

=== et/GLEAMv4.3b ===


/glade/u/home/bbuchovecky/projects/et_unc/etunc/load/obs.py:621: UserWarning: gleam v4.3b monthly E: no files for years [2001, 2002]
  warnings.warn(f"{spec.name} {version} {freq} {var}: no files for years {missing}")



=== et/SiTHv2 ===

=== et/FLUXCOM-X-BASE ===

=== et/FLUXCOM ===

=== lai/GIMMS_LAI4g ===

=== pr/GPCPv3.3 ===

=== rn/CERESed4.2 ===


In [4]:
ann["lai"]["GIMMS_LAI4g"]

<xarray.DataArray 'lai' (time: 168, lat: 360, lon: 720)> Size: 348MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], shape=(168, 360, 720))
Coordinates:
  * time     (time) object 1kB 2001-01-01 00:00:00 ... 2014-12-01 00:00:00
  * lat      (lat) float32 1kB -89.75 -89.25 -88.75 -88.25 ... 88.75 89.25 89.75
  * lon      (lon) float32 3kB -179.8 -179.2 -178.8 -178.2 ... 178.8 179.2 179.8
Attributes:
    long_name:  leaf_area_index
    units:      1
    source:     ilamb